# Hands-on: from SEM micrographs to FAIR numbers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bashirkazimi/caden2026-ml-handson/blob/main/notebooks/01_ma_segmentation.ipynb)

**NFDI4ING summer school "Practical Research Data Management in Materials Science"**
IMD-2, Forschungszentrum Jülich, 14 October 2026, Bashir Kazimi (IAS-9)

You will segment **martensite–austenite (M–A) islands** in SEM images of a bainitic
steel, measure the **M–A area fraction** (the number a materials scientist actually
wants), and save the analysis so that someone else can **find, understand and
reproduce** it.

| Part | What you do | Time |
|---|---|---|
| 0 | Setup: get the data, check its integrity | 2 min |
| 1 | Look at the data **and its metadata** | 5 min |
| 2 | Classical baseline: a global threshold (Otsu) | 3 min |
| 3 | Choose how to split the data (the most important decision today) | 2 min |
| 4 | Train a small U-Net (seconds on a GPU, ~1 min on a laptop CPU) | 4 min |
| 5 | Evaluate: pixels *and* materials numbers | 4 min |
| 6 | Make the result FAIR: config, provenance, model card, RO-Crate | 5 min |
| 7 | Take-home exercises | at home |

Run the cells from top to bottom with **Shift + Enter**. Sections marked **Exercise**
contain a small task.

**On Google Colab** nothing needs to be installed. The notebook asks Colab for a GPU;
if the first code cell reports `device: cpu`, choose *Runtime → Change runtime type →
T4 GPU* and run it again. A CPU works too, training just takes about a minute.

**Data:** Ackermann, Iren, Wesselmecking, Krupp, Shetty (2022), *M-A island segmentation
of bainite microstructures in SEM image data*, figshare,
[doi:10.6084/m9.figshare.19232523.v2](https://doi.org/10.6084/m9.figshare.19232523.v2)
(MIT licence); paper: *Materials Characterization* 191, 112091
([doi:10.1016/j.matchar.2022.112091](https://doi.org/10.1016/j.matchar.2022.112091)).
We use a 2× downsampled teaching subset (40 images, 22 MB).

## Part 0: Setup

The next cell imports the libraries and fixes **all random seeds**. A result you
cannot regenerate is not a result.

In [ ]:
import csv
import hashlib
import json
import platform
import random
import shutil
import sys
import time
import urllib.request
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)  # numpy: we pass explicit np.random.default_rng(seed) generators

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python {platform.python_version()}, torch {torch.__version__}, numpy {np.__version__}")
print(f"device: {DEVICE}, CPU threads used by torch: {torch.get_num_threads()}")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and DEVICE == "cpu":
    print("Tip: Runtime -> Change runtime type -> T4 GPU makes training much faster (optional).")

Get the data. The cell looks for the teaching dataset next to the notebook, downloads
it if needed, and **verifies every file against its SHA-256 checksum**, the same idea
a repository like Zenodo uses to guarantee that what you downloaded is what was
published. If nothing can be found or downloaded, it falls back to a *synthetic*
look-alike dataset so the session still runs (you will be told).

In [ ]:
DATA_NAME = "ma_bainite_teaching"
DATA_URL = (
    "https://raw.githubusercontent.com/bashirkazimi/caden2026-ml-handson/main/"
    "data/ma_bainite_teaching.zip"
)
SEARCH = [Path("data"), Path("../data"), Path("hands-on/data"), Path("caden2026-ml-handson/data")]


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk)
            if not block:
                return h.hexdigest()
            h.update(block)


def verify_checksums(root):
    lines = (root / "checksums.sha256").read_text().split("\n")
    bad = []
    for line in filter(None, lines):
        digest, name = line.split("  ", 1)
        if sha256(root / name) != digest:
            bad.append(name)
    n_files = len([line for line in lines if line])
    print(f"checksums: {n_files - len(bad)} of {n_files} files OK, {len(bad)} corrupted")
    return not bad


def _smooth_field(rng, h, w, sigma):
    ky, kx = np.fft.fftfreq(h)[:, None], np.fft.fftfreq(w)[None, :]
    g = np.exp(-2 * (np.pi * sigma) ** 2 * (kx**2 + ky**2))
    return np.real(np.fft.ifft2(np.fft.fft2(rng.standard_normal((h, w))) * g))


def make_synthetic_dataset(root, h=384, w=512, n_per_series=10):
    """Fallback only: blob 'microstructures' with the same file layout as the real data."""
    rng = np.random.default_rng(SEED)
    (root / "images").mkdir(parents=True, exist_ok=True)
    (root / "masks").mkdir(exist_ok=True)
    rows = []
    series = [("syn-a", "i", 0.12, 4), ("syn-b", "i", 0.16, 5), ("syn-c", "ii", 0.25, 7), ("syn-d", "iii", 0.3, 9)]
    for name, strategy, frac, sigma in series:
        for k in range(n_per_series):
            field = _smooth_field(rng, h, w, sigma)
            mask = field > np.quantile(field, 1 - frac)
            tex = _smooth_field(rng, h, w, 1.5)
            img = 0.5 + 0.2 * mask + 0.1 * tex / tex.std() + 0.06 * rng.standard_normal((h, w))
            img = np.clip(img + 0.1 * rng.uniform(-1, 1) * np.linspace(-1, 1, w)[None, :], 0, 1)
            iid = f"{strategy}__{name}-{k:02d}"
            Image.fromarray((img * 255).astype(np.uint8)).save(root / "images" / f"{iid}.png")
            Image.fromarray((mask * 255).astype(np.uint8)).save(root / "masks" / f"{iid}.png")
            rows.append(dict(image_id=iid, image_file=f"images/{iid}.png", mask_file=f"masks/{iid}.png",
                             cooling_strategy=strategy, sample_series=name, section="?", position_mm="?",
                             original_split="train", ma_area_fraction=round(float(mask.mean()), 4),
                             pixel_size_nm=""))
    with open(root / "metadata.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return root


def get_data():
    for base in SEARCH:
        if (base / DATA_NAME / "metadata.csv").exists():
            return base / DATA_NAME, "real"
    for base in SEARCH:
        archive = base / f"{DATA_NAME}.zip"
        if archive.exists() and zipfile.is_zipfile(archive):
            zipfile.ZipFile(archive).extractall(base)
            return base / DATA_NAME, "real"
    archive = Path("data") / f"{DATA_NAME}.zip"
    try:
        archive.parent.mkdir(exist_ok=True)
        print("downloading", DATA_URL)
        urllib.request.urlretrieve(DATA_URL, archive)
        zipfile.ZipFile(archive).extractall("data")
        return Path("data") / DATA_NAME, "real"
    except Exception as err:  # offline, or the URL is not public yet
        archive.unlink(missing_ok=True)
        print(f"download failed ({err}) -> using a SYNTHETIC stand-in dataset")
        return make_synthetic_dataset(Path("data") / "synthetic_stand_in"), "synthetic"


DATA_DIR, DATA_KIND = get_data()
print("data:", DATA_DIR.resolve(), f"({DATA_KIND})")
if DATA_KIND == "real":
    assert verify_checksums(DATA_DIR), "some files are corrupted - download again"

## Part 1: Look at the data *and its metadata*

Every image has one row in `metadata.csv`. Let's read it and count what we have.

In [ ]:
with open(DATA_DIR / "metadata.csv") as f:
    ROWS = list(csv.DictReader(f))

print(f"{len(ROWS)} images; columns:\n  " + ", ".join(ROWS[0].keys()))
print()
for key in ("cooling_strategy", "sample_series"):
    counts = {}
    for r in ROWS:
        counts[r[key]] = counts.get(r[key], 0) + 1
    print(f"{key:>17}: {counts}")
print("\nexample row:")
for k, v in ROWS[0].items():
    print(f"  {k:>20} = {v!r}")

Load the images (as floats in [0, 1]) and the masks (True = M–A island). To keep
training fast on a CPU we work at half the resolution of the teaching subset
(`SCALE = 0.5` → 512 × 384 px).

In [ ]:
SCALE = 0.5


def load_pair(row, scale=SCALE):
    img = Image.open(DATA_DIR / row["image_file"]).convert("L")
    msk = Image.open(DATA_DIR / row["mask_file"]).convert("L")
    if scale != 1:
        size = (round(img.width * scale), round(img.height * scale))
        img, msk = img.resize(size, Image.BOX), msk.resize(size, Image.BOX)
    return np.asarray(img, np.float32) / 255, np.asarray(msk) >= 128


DATA = {r["image_id"]: load_pair(r) for r in ROWS}
print(f"loaded {len(DATA)} image/mask pairs of shape {next(iter(DATA.values()))[0].shape}")


def overlay(img, mask, color=(1.0, 0.5, 0.0), alpha=0.45):
    rgb = np.repeat(img[..., None], 3, axis=2)
    rgb[mask] = (1 - alpha) * rgb[mask] + alpha * np.array(color)
    return rgb


examples = [next(r for r in ROWS if r["cooling_strategy"] == s) for s in ("i", "ii", "iii")]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for col, r in enumerate(examples):
    img, msk = DATA[r["image_id"]]
    axes[0, col].imshow(img, cmap="gray")
    axes[0, col].set_title(f"strategy {r['cooling_strategy']}, {r['sample_series']}", fontsize=10)
    axes[1, col].imshow(overlay(img, msk))
    axes[1, col].set_title(f"M–A area fraction = {msk.mean():.1%}", fontsize=10)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Exercise 1: what does the metadata *not* tell you? (discuss with your neighbour)

Look at the example row above and at the column `pixel_size_nm`.

1. Which experimental information is encoded **only in the file name**? Do you know
   what `H`/`V` and `6mm`/`9mm`/`12mm` mean?
2. Can you report the **mean island size in µm²** from these data? Why (not)?
3. If *you* had recorded these images and documented them in an ELN such as eLabFTW,
   which fields would you have added so that a stranger could reuse them?

*Write your notes here.*

## Part 2: Classical baseline: one global threshold (Otsu)

M–A islands look bright. The textbook approach is to pick one grey-value threshold per
image that best separates two populations (Otsu's method) and call everything brighter
"M–A". No training data needed. How well does that work?

In [ ]:
def otsu_threshold(img):
    hist, edges = np.histogram(img, bins=256, range=(0, 1))
    p = hist / hist.sum()
    w = np.cumsum(p)
    mu = np.cumsum(p * np.arange(256))
    between_class_variance = (mu[-1] * w - mu) ** 2 / (w * (1 - w) + 1e-12)
    return edges[np.argmax(between_class_variance) + 1]


def dice(pred, true):
    """Overlap score: 1 = perfect, 0 = no overlap."""
    return 2 * (pred & true).sum() / (pred.sum() + true.sum() + 1e-9)


def area_fraction(mask):
    """Fraction of pixels labelled M-A. Question: why is this valid even without a pixel size?"""
    return float(mask.mean())


OTSU = {}
for r in ROWS:
    img, true = DATA[r["image_id"]]
    pred = img > otsu_threshold(img)
    OTSU[r["image_id"]] = pred
    r["otsu_dice"] = dice(pred, true)
    r["otsu_fraction_error_pp"] = 100 * (area_fraction(pred) - area_fraction(true))

for s in ("i", "ii", "iii"):
    sel = [r for r in ROWS if r["cooling_strategy"] == s]
    print(f"strategy {s:>3}: mean Dice = {np.mean([r['otsu_dice'] for r in sel]):.2f}, "
          f"mean |area-fraction error| = {np.mean([abs(r['otsu_fraction_error_pp']) for r in sel]):4.1f} "
          f"percentage points")

A single threshold over-counts M–A badly: the matrix contains bright relief and
texture of the same grey level. We need a method that uses **shape and context**, not
only brightness → a convolutional neural network.

## Part 3: Decide how to split the data

We will train on some images and test on others. *Which* images end up in the test set
decides what your final number means:

- `"random"`: 10 random images for testing, 30 for training. Images of the **same
  sample series** end up on both sides.
- `"hold_out_i"`, `"hold_out_ii"`, `"hold_out_iii"`: test on **all images of one
  cooling strategy** that the model has never seen; train on the other two. This is
  the question "does it work on a *new material*?".

**Exercise:** run everything once with the default split, `"hold_out_iii"`. Then change
`"split"` in the next cell to `"random"` or `"hold_out_i"`, rerun from that cell onwards
(Colab: *Runtime → Run cell and below*; JupyterLab: *Run → Run Selected Cell and All
Below*) and compare Dice and area-fraction error with your first run.

In [ ]:
CONFIG = {
    "split": "hold_out_iii",  # choose: "random" | "hold_out_i" | "hold_out_ii" | "hold_out_iii"
    "seed": SEED,
    "scale": SCALE,
    "crop": 128,  # training patches of 128 x 128 px
    "batch_size": 8,
    "iterations": 300,
    "learning_rate": 2e-3,
    "unet_width": 16,  # channels in the first layer
    "threshold": 0.5,
}


def make_split(rows, split, seed):
    if split == "random":
        order = np.random.default_rng(seed).permutation(len(rows))
        test = [rows[i] for i in order[:10]]
    elif split.startswith("hold_out_"):
        held = split[len("hold_out_"):]
        test = [r for r in rows if r["cooling_strategy"] == held]
    else:
        raise ValueError(f"unknown split {split!r}")
    test_ids = {r["image_id"] for r in test}
    train = [r for r in rows if r["image_id"] not in test_ids]
    return train, test


TRAIN, TEST = make_split(ROWS, CONFIG["split"], CONFIG["seed"])
print(f"split = {CONFIG['split']}: {len(TRAIN)} training / {len(TEST)} test images")
shared = {r["sample_series"] for r in TRAIN} & {r["sample_series"] for r in TEST}
print("sample series in BOTH train and test:", shared or "none")

## Part 4: Train a small U-Net

A U-Net is the workhorse of scientific image segmentation: an **encoder** that
compresses the image into increasingly abstract features, a **decoder** that expands
them back to full resolution, and **skip connections** that carry fine detail across.
Ours has ~0.5 million parameters: tiny by modern standards, but enough for this task.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, width=16, depth=3):
        super().__init__()
        ch = [width * 2**i for i in range(depth + 1)]
        self.down = nn.ModuleList([conv_block(1 if i == 0 else ch[i - 1], ch[i]) for i in range(depth)])
        self.bottom = conv_block(ch[depth - 1], ch[depth])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, stride=2) for i in reversed(range(depth))])
        self.merge = nn.ModuleList([conv_block(2 * ch[i], ch[i]) for i in reversed(range(depth))])
        self.head = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottom(x)
        for up, merge, skip in zip(self.up, self.merge, reversed(skips)):
            x = merge(torch.cat([up(x), skip], dim=1))
        return self.head(x)  # logits; sigmoid -> probability of "M-A"


model = UNet(CONFIG["unet_width"]).to(DEVICE)
print(f"U-Net with {sum(p.numel() for p in model.parameters()):,} trainable parameters")

Training ingredients: random **128 × 128 crops** with rotations/flips (a micrograph has
no preferred orientation, unlike a diffraction pattern), each image
**standardised** to zero mean and unit variance (removes brightness/contrast
differences between sessions), and a loss = binary cross-entropy + Dice loss.

Run the next cell. While it trains (~1 min on a CPU), think about Exercise 1 again.

In [ ]:
def standardise(img):
    return (img - img.mean()) / (img.std() + 1e-6)


def random_batch(pairs, batch_size, crop, rng):
    xs, ys = [], []
    for _ in range(batch_size):
        img, msk = pairs[rng.integers(len(pairs))]
        y0, x0 = rng.integers(img.shape[0] - crop + 1), rng.integers(img.shape[1] - crop + 1)
        a, b = img[y0:y0 + crop, x0:x0 + crop], msk[y0:y0 + crop, x0:x0 + crop]
        k = rng.integers(4)
        a, b = np.rot90(a, k), np.rot90(b, k)
        if rng.random() < 0.5:
            a, b = a[:, ::-1], b[:, ::-1]
        xs.append(a.copy())
        ys.append(b.copy())
    return (torch.from_numpy(np.stack(xs)[:, None]).float().to(DEVICE),
            torch.from_numpy(np.stack(ys)[:, None]).float().to(DEVICE))


def loss_fn(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    soft_dice = (2 * (p * target).sum() + 1) / (p.sum() + target.sum() + 1)
    return bce + (1 - soft_dice)


torch.manual_seed(CONFIG["seed"])
rng = np.random.default_rng(CONFIG["seed"])
model = UNet(CONFIG["unet_width"]).to(DEVICE)
optimiser = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, max_lr=CONFIG["learning_rate"],
                                               total_steps=CONFIG["iterations"])
train_pairs = [(standardise(DATA[r["image_id"]][0]), DATA[r["image_id"]][1]) for r in TRAIN]

losses, t0 = [], time.time()
for it in range(1, CONFIG["iterations"] + 1):
    model.train()
    x, y = random_batch(train_pairs, CONFIG["batch_size"], CONFIG["crop"], rng)
    loss = loss_fn(model(x), y)
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    schedule.step()
    losses.append(loss.item())
    if it % 50 == 0 or it == 1:
        print(f"iteration {it:4d}/{CONFIG['iterations']}, loss {loss.item():.3f}, {time.time() - t0:5.1f} s")
TRAIN_SECONDS = time.time() - t0

plt.figure(figsize=(6, 3))
plt.plot(losses, lw=0.8)
plt.xlabel("iteration")
plt.ylabel("training loss")
plt.title(f"trained in {TRAIN_SECONDS:.0f} s on {DEVICE}")
plt.tight_layout()
plt.show()

## Part 5: Evaluate: pixels *and* materials numbers

Two kinds of metric:

- **Dice**: how well do predicted and true M–A pixels overlap? (computer-vision view)
- **Area-fraction error** in percentage points: how wrong is the *phase fraction* you
  would put in your paper? (materials view)

In [ ]:
@torch.no_grad()
def predict(model, img):
    model.eval()
    h, w = img.shape
    padded = np.pad(standardise(img), ((0, (-h) % 8), (0, (-w) % 8)), mode="reflect")
    prob = torch.sigmoid(model(torch.from_numpy(padded)[None, None].float().to(DEVICE)))
    return prob[0, 0, :h, :w].cpu().numpy()


RESULTS = []
PRED = {}
for r in TEST:
    img, true = DATA[r["image_id"]]
    pred = predict(model, img) > CONFIG["threshold"]
    PRED[r["image_id"]] = pred
    RESULTS.append({
        "image_id": r["image_id"],
        "cooling_strategy": r["cooling_strategy"],
        "dice_unet": round(dice(pred, true), 4),
        "dice_otsu": round(r["otsu_dice"], 4),
        "fraction_true": round(area_fraction(true), 4),
        "fraction_unet": round(area_fraction(pred), 4),
        "fraction_otsu": round(area_fraction(OTSU[r["image_id"]]), 4),
    })

print(f"{'image':<34} {'Dice U-Net':>10} {'Dice Otsu':>10} {'f_true':>7} {'f_U-Net':>8} {'f_Otsu':>7}")
for m in RESULTS:
    print(f"{m['image_id']:<34} {m['dice_unet']:>10.3f} {m['dice_otsu']:>10.3f} "
          f"{m['fraction_true']:>7.1%} {m['fraction_unet']:>8.1%} {m['fraction_otsu']:>7.1%}")

SUMMARY = {
    "split": CONFIG["split"],
    "mean_dice_unet": round(float(np.mean([m["dice_unet"] for m in RESULTS])), 3),
    "mean_dice_otsu": round(float(np.mean([m["dice_otsu"] for m in RESULTS])), 3),
    "mean_abs_fraction_error_pp_unet": round(float(np.mean(
        [100 * abs(m["fraction_unet"] - m["fraction_true"]) for m in RESULTS])), 2),
    "mean_abs_fraction_error_pp_otsu": round(float(np.mean(
        [100 * abs(m["fraction_otsu"] - m["fraction_true"]) for m in RESULTS])), 2),
}
print(f"\nResult for split {CONFIG['split']}: Dice = {SUMMARY['mean_dice_unet']:.2f}, "
      f"area-fraction error = {SUMMARY['mean_abs_fraction_error_pp_unet']:.1f} pp")

Look at one test image: where does the network make mistakes?
(red = predicted M–A that is not in the label; blue = missed M–A)

In [ ]:
r = TEST[0]
img, true = DATA[r["image_id"]]
pred = PRED[r["image_id"]]
errors = np.repeat(img[..., None], 3, axis=2) * 0.6
errors[pred & ~true] = (0.9, 0.1, 0.1)
errors[~pred & true] = (0.1, 0.3, 0.9)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
panels = [(img, "SEM image", "gray"), (overlay(img, true), "label (annotator)", None),
          (overlay(img, OTSU[r["image_id"]]), f"Otsu (Dice {dice(OTSU[r['image_id']], true):.2f})", None),
          (errors, f"U-Net errors (Dice {dice(pred, true):.2f})", None)]
for ax, (im, title, cmap) in zip(axes, panels):
    ax.imshow(im, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ft = [m["fraction_true"] for m in RESULTS]
ax.plot([0, 0.7], [0, 0.7], "k--", lw=0.8, label="perfect")
ax.scatter(ft, [m["fraction_otsu"] for m in RESULTS], label="Otsu", marker="s", alpha=0.7)
ax.scatter(ft, [m["fraction_unet"] for m in RESULTS], label="U-Net", alpha=0.8)
ax.set_xlabel("true M–A area fraction")
ax.set_ylabel("measured M–A area fraction")
ax.legend()
plt.tight_layout()
plt.show()

### Exercise 2: compare splits

Run the notebook with at least two splits (the default `"hold_out_iii"`, then `"random"`
or `"hold_out_i"`, see Part 3) and note Dice and area-fraction error for each. Then think
about:

1. Which split gives the most *optimistic* numbers? Which one answers the question you
   actually care about ("will this work on my next sample")?
2. Is Dice comparable between `hold_out_i` and `hold_out_iii`? (Hint: look at the true
   area fractions of the two strategies.)

*Notes:*

## Part 6: Make the result FAIR

Your Dice score is worthless next month if nobody (including you) can tell **which
data, which split, which code, which settings** produced it. The next cell writes a
self-describing *run folder*:

```
runs/<run_id>/
  config.json            every setting above
  provenance.json        data checksum + source DOI, software versions, seed, split IDs
  metrics.csv            per-image results
  model.pt               weights + config + provenance in one file
  MODEL_CARD.md          human-readable summary: intended use, limitations
  ro-crate-metadata.json machine-readable package description (RO-Crate 1.1)
```

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + f"_{CONFIG['split']}"
RUN = Path("runs") / RUN_ID
RUN.mkdir(parents=True, exist_ok=True)

dataset_provenance = {}
if (DATA_DIR / "provenance.json").exists():
    dataset_provenance = json.loads((DATA_DIR / "provenance.json").read_text())

PROVENANCE = {
    "run_id": RUN_ID,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "01_ma_segmentation.ipynb",
    "data": {
        "kind": DATA_KIND,
        "name": DATA_DIR.name,
        "metadata_csv_sha256": sha256(DATA_DIR / "metadata.csv"),
        "source": dataset_provenance.get("source", {}),
        "train_ids": [r["image_id"] for r in TRAIN],
        "test_ids": [r["image_id"] for r in TEST],
    },
    "software": {"python": platform.python_version(), "torch": torch.__version__,
                 "numpy": np.__version__, "platform": platform.platform()},
    "hardware": {"device": DEVICE, "torch_threads": torch.get_num_threads()},
    "training_seconds": round(TRAIN_SECONDS, 1),
    "seed": CONFIG["seed"],
}

(RUN / "config.json").write_text(json.dumps(CONFIG, indent=2))
(RUN / "provenance.json").write_text(json.dumps(PROVENANCE, indent=2))
with open(RUN / "metrics.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(RESULTS[0]))
    writer.writeheader()
    writer.writerows(RESULTS)
torch.save({"state_dict": model.state_dict(), "config": CONFIG, "provenance": PROVENANCE,
            "summary": SUMMARY}, RUN / "model.pt")
print("written:", *sorted(p.name for p in RUN.iterdir()), sep="\n  ")

### Exercise 3: write the model card

A **model card** tells the next person what the model is for and, more importantly,
what it is *not* for. Fill in the two fields below, then run the cell.

In [ ]:
INTENDED_USE = "TODO: for which images / which question may this model be used?"
LIMITATIONS = "TODO: when should nobody trust it? (data, labels, imaging conditions, metrics)"

card = f"""# Model card: U-Net for M-A islands ({RUN_ID})

## Model
Small U-Net (width {CONFIG['unet_width']}, depth 3, {sum(p.numel() for p in model.parameters()):,} parameters),
trained {CONFIG['iterations']} iterations on {CONFIG['crop']}x{CONFIG['crop']} crops, seed {CONFIG['seed']},
{TRAIN_SECONDS:.0f} s on {DEVICE}. Full settings: `config.json`.

## Data
{DATA_DIR.name} ({DATA_KIND}); source DOI: {dataset_provenance.get('source', {}).get('doi', 'n/a')} (MIT).
Split `{CONFIG['split']}`: {len(TRAIN)} training / {len(TEST)} test images (IDs in `provenance.json`).

## Results on the test images
| metric | U-Net | Otsu baseline |
|---|---|---|
| mean Dice | {SUMMARY['mean_dice_unet']:.3f} | {SUMMARY['mean_dice_otsu']:.3f} |
| mean abs. area-fraction error (pp) | {SUMMARY['mean_abs_fraction_error_pp_unet']:.2f} | {SUMMARY['mean_abs_fraction_error_pp_otsu']:.2f} |

## Intended use
{INTENDED_USE}

## Limitations
{LIMITATIONS}
"""
(RUN / "MODEL_CARD.md").write_text(card)
print(card)

Finally, describe the run folder in **RO-Crate**, a small JSON-LD file that tells
machines what each file is, who/what made it, and under which licence. RO-Crate is the
basis of the `.eln` exchange format that ELNs such as eLabFTW and Kadi4Mat use to
move experiments between systems.

In [ ]:
files = sorted(p.name for p in RUN.iterdir() if p.name != "ro-crate-metadata.json")
crate = {
    "@context": "https://w3id.org/ro/crate/1.1/context",
    "@graph": [
        {"@id": "ro-crate-metadata.json", "@type": "CreativeWork",
         "conformsTo": {"@id": "https://w3id.org/ro/crate/1.1"}, "about": {"@id": "./"}},
        {"@id": "./", "@type": "Dataset", "name": f"U-Net M-A segmentation run {RUN_ID}",
         "description": "Segmentation of M-A islands in SEM images of bainitic steel (teaching run).",
         "datePublished": PROVENANCE["created_utc"], "license": {"@id": "https://spdx.org/licenses/MIT"},
         "isBasedOn": {"@id": "https://doi.org/10.6084/m9.figshare.19232523.v2"},
         "hasPart": [{"@id": name} for name in files]},
        {"@id": "https://doi.org/10.6084/m9.figshare.19232523.v2", "@type": "Dataset",
         "name": "M-A island segmentation of bainite microstructures in SEM image data"},
        *[{"@id": name, "@type": "File", "contentSize": (RUN / name).stat().st_size,
           "sha256": sha256(RUN / name)} for name in files],
    ],
}
(RUN / "ro-crate-metadata.json").write_text(json.dumps(crate, indent=2))
print(f"run folder ready: {RUN.resolve()}")
for p in sorted(RUN.iterdir()):
    print(f"  {p.name:<26} {p.stat().st_size / 1024:8.1f} kB")

**Keep a copy.** On Colab, files disappear when the session ends. The next cell zips the
run folder; on Colab it also downloads the zip to your computer.

In [ ]:
archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN.parent, base_dir=RUN.name)
print("archive:", archive)
if IN_COLAB:
    from google.colab import files

    files.download(archive)

**Where would this go?** Attach the run folder to the eLabFTW experiment in which the
images were recorded (or a Kadi4Mat record), so that sample → images → labels → model →
numbers form one linked chain. The sketch below shows the idea with the eLabFTW REST
API v2; it is **not executed**: adapt the URL and use your own API key, and check your
instance's API documentation.

```python
import requests
API = "https://<your-elabftw-instance>/api/v2"
HEADERS = {"Authorization": "<your API key>"}
r = requests.post(f"{API}/experiments", headers=HEADERS, json={"title": f"U-Net M-A run {RUN_ID}"})
exp_id = r.headers["Location"].rstrip("/").split("/")[-1]
requests.patch(f"{API}/experiments/{exp_id}", headers=HEADERS, json={"body": card})
for path in RUN.iterdir():
    with open(path, "rb") as fh:
        requests.post(f"{API}/experiments/{exp_id}/uploads", headers=HEADERS, files={"file": fh})
```

## Part 7: Take-home exercises

1. **How many labels do you need?** Train on only 1, 2, 5, 10 images of the training
   set (keep the test set fixed). Plot Dice and area-fraction error against the number
   of training images. *(When we tried this, ~5 densely labelled images got close to
   the 30-image result, but single images were erratic.)*
2. **Remove the standardisation** in `random_batch`/`predict` and train with
   `hold_out_i`. What happens, and why does it matter when images come from different
   sessions or microscopes?
3. **Seeds.** Re-run with `seed` = 1 and 2. How large is the run-to-run spread compared
   with the difference between splits?
4. **Your own data.** Which of your own measurements (for example XRD patterns or Raman
   spectra, if you recorded some during this summer school) could be analysed the same
   way, and what metadata would a model need from their ELN record?
   (See `02_bonus_spectra.ipynb` for a 1D version of this workflow.)